The first part of this series, [Random fields and the sum-of-sinusoids method](tutorials/sos_01_random_fields.html), used the presets of Quadriga-Lib: tables of frequencies fitted to three autocorrelation functions (ACFs). This tutorial shows how to make such a table for an ACF of your own, how to judge it, and where the sum-of-sinusoids method reaches its limits: correlation functions that cannot exist, the error against the number of sinusoids, the kink of the exponential ACF at the origin, and values that are not Normal distributed. The last example does without a fit and builds a traveling sea surface from frequencies drawn from a wave spectrum, with time as a coordinate.

The tutorial builds on part 1: the ACF, the decorrelation distance, presets, phases and seeds are used as introduced there. It needs Quadriga-Lib 0.13 or newer, numpy and matplotlib; see the setup on the [tutorial overview](tutorials.html#setup).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import quadriga_lib as qd

print(qd.version())

## A field with two length scales

As an example, take a quantity that varies on two scales: 60 % of its variance changes within about 5 m, the other 40 % within about 50 m. Its ACF is a weighted sum of two exponentials, $\rho(d) = 0.6\, e^{-d/5\,\mathrm{m}} + 0.4\, e^{-d/50\,\mathrm{m}}$, and no preset has it. `sos.generate` fits 256 frequencies to it in a few seconds, and `sos.map` evaluates them on a grid, as in part 1. The sample points end at 250 m, five times the longer scale, as the presets end at five decorrelation distances. How they are spaced and the frequency limits passed here are explained in the next two sections.

In [ ]:
dist = np.r_[0.0, np.geomspace(0.1, 250.0, 200)]                  # sample points in m, dense at short distances
acf = 0.6 * np.exp(-dist / 5.0) + 0.4 * np.exp(-dist / 50.0)      # target ACF with length scales of 5 m and 50 m
freq, _, _ = qd.sos.generate(acf, dist, 256, n_dim=3, n_dir=20, freq_min=0.001, freq_max=1.0, seed=1)   # [256, 3] in 1/m

x = np.arange(0.0, 300.0, 0.5)                                     # grid points in m, x and y
field = qd.sos.map(freq, qd.sos.init_phases(256, 1, 4), x, x)[:, :, 0, 0]
plt.imshow(field, origin="lower", extent=[0, 300, 0, 300], cmap="viridis")
plt.colorbar(label="value")
plt.xlabel("x in m"); plt.ylabel("y in m")
plt.show()

## The frequencies are the free parameters

A generator with $N$ sinusoids has the ACF

$$\hat\rho(\mathbf{d}) = \frac{1}{N} \sum_{n=1}^{N} \cos\!\left( 2\pi\, \mathbf{f}_n \cdot \mathbf{d} \right)$$

for random phases (part 1). All sinusoids have the same amplitude $\sqrt{2/N}$, so the model has no weights: the frequency vectors $\mathbf{f}_n$ are its only free parameters. A strong part of the spectrum is represented by many frequencies close together, a weak part by few. The frequencies of the field above show the two scales: a narrow group near $1/(2\pi \cdot 50\,\mathrm{m}) \approx 0.003$ 1/m for the 50 m scale, and a broad group around $1/(2\pi \cdot 5\,\mathrm{m}) \approx 0.03$ 1/m that reaches up to the limit of 1 1/m for the 5 m scale. Only about a quarter of the frequencies lie below 0.01 1/m, fewer than the 40 % share of the 50 m scale in the variance, because the spectrum of an exponential ACF reaches far beyond its peak.

In [ ]:
f_abs = np.linalg.norm(freq, axis=1)                               # magnitude of each frequency vector in 1/m
plt.figure(figsize=(6.4, 3))
plt.hist(f_abs, bins=np.geomspace(1e-3, 1.0, 40), alpha=0.7)
plt.xscale("log"); plt.xlabel("|f| in 1/m"); plt.ylabel("number of sinusoids")
plt.show()
print(f"below 0.01 1/m: {np.mean(f_abs < 0.01):.0%} of the sinusoids")

`sos.generate` is based on the method of [1, Sec. III]: each sinusoid has a fixed direction, the frequency of one sinusoid after the other is refitted to what the other sinusoids leave of the target, and the new frequency is kept only if the mean squared error over a set of test directions decreases; the passes stop when nothing improves. In the library, the directions are spread regularly over a half circle (2-D) or a half sphere (3-D), and each frequency is fitted along the sinusoid's own direction. The arguments control what is fitted:

- `acf` and `dist`: the target at the sample points. The fit controls the ACF only up to the last sample point. Each sample point counts equally in the error, so their spacing decides where the fit is accurate.
- The number of sinusoids. More sinusoids approximate better and cost proportionally more time when the field is evaluated.
- `n_dim`: 1, 2 or 3 dimensions. A 3-D table also works for positions on a line or in a plane.
- `n_dir`: the number of test directions. With few of them, the fit is better in those directions than in others; the presets were made with 300. This notebook mostly uses 20 to keep the fits fast.
- `freq_min` and `freq_max`: limits of the frequency magnitude. By default there is no lower limit, and the upper limit is $15/\max(\mathrm{dist})$. The presets use the lower limit $0.05/d_0$; for the 50 m scale above this is 0.001 1/m.
- `seed`: makes the result repeatable.

## Judging a fit

The sample points and limits of the first cell are not the defaults. With the defaults (equally spaced sample points, no lower limit and the default upper limit) the fit has a small mean error, but it has lost the short scale. `sos.calc_mse` returns the error of a table as $-10 \log_{10}$ of the mean squared error, so larger is better, in three ways: along test directions at the sample points (`mse`); on a disk in the planes of the coordinate axes (`mse_core`); and on a grid in these planes twice as large, whose target is 0 beyond the last sample point (`mse_all`). It also returns `acf_tail`, the ACF of the generator at 2, 4, 10 and 20 times the last sample distance, where the target is 0. It needs equally spaced sample points, so the cell uses the 251 points of the default fit, 1 m apart:

In [ ]:
two_scales = lambda d: 0.6 * np.exp(-d / 5.0) + 0.4 * np.exp(-d / 50.0)
d_lin = np.arange(0.0, 250.1, 1.0)                                  # 251 equally spaced sample points
freq_def, _, _ = qd.sos.generate(two_scales(d_lin), d_lin, 256, n_dim=3, n_dir=20, seed=1)   # default limits

for name, f in (("default", freq_def), ("tuned", freq)):
    mse, mse_core, mse_all, tail, _ = qd.sos.calc_mse(f, two_scales(d_lin), d_lin, n_dir=100)
    print(f"{name:8s} mse {mse:5.1f} dB   mse_core {mse_core:5.1f} dB   mse_all {mse_all:5.1f} dB   acf_tail {np.round(tail, 3)}")

The tuned table is a little better along the test directions, and on the disk and the grid the two hardly differ. The disk and the grid are areas, in which the first meters are a small part, and the mean over the test directions counts the first meters no more than the rest. None of the numbers says where the error sits. The ACF itself shows it. The cell evaluates the formula above along 200 random directions, plots the mean over the directions with a band that holds the middle 90 % of the directions, and prints the error of the mean:

In [ ]:
rng = np.random.default_rng(5)
u = rng.standard_normal((200, 3)); u /= np.linalg.norm(u, axis=1, keepdims=True)     # 200 random directions
def acf_along(f, d):
    # ACF of a table f [N, n_dim] at the distances d [n_d] along the 200 directions (in n_dim dimensions); [200, n_d]
    ud = u[:, :f.shape[1]] / np.linalg.norm(u[:, :f.shape[1]], axis=1, keepdims=True)
    return np.array([np.cos(2 * np.pi * np.outer(f @ v, d)).mean(axis=0) for v in ud])

d_plot = np.linspace(0.0, 250.0, 1001)
acfs = {"default": acf_along(freq_def, d_plot), "tuned": acf_along(freq, d_plot)}
fig, axs = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
for ax, xmax in zip(axs, (20, 250)):
    ax.plot(d_plot, two_scales(d_plot), "k--", lw=2, label="target")
    for name, col in (("default", "C3"), ("tuned", "C0")):
        ax.fill_between(d_plot, *np.percentile(acfs[name], [5, 95], axis=0), color=col, alpha=0.2, lw=0, label=f"{name}, 90 % of directions")
        ax.plot(d_plot, acfs[name].mean(axis=0), color=col, label=f"{name}, mean")
    ax.set_xlim(0, xmax); ax.set_xlabel("distance d in m"); ax.grid(alpha=0.3)
axs[0].set_ylabel("ACF"); axs[0].legend(fontsize=8)
plt.show()
for name, a in acfs.items():
    e = a.mean(axis=0) - two_scales(d_plot)
    print(f"{name:8s} error of the mean ACF: rms {np.sqrt(np.mean(e**2)):.3f}, largest {np.abs(e).max():.3f}; ACF at 1, 2, 5 m: "
          + ", ".join(f"{np.interp(d, d_plot, a.mean(axis=0)):.2f}" for d in (1, 2, 5)))
print(f"target   ACF at 1, 2, 5 m: " + ", ".join(f"{two_scales(d):.2f}" for d in (1, 2, 5)))

The default table is too smooth at short distances: its ACF stays near 1 over the first meters, where the target falls to 0.88 within the first meter. Two settings cause this. With 251 equally spaced sample points, only 10 of them lie in the first 10 m, so the short scale hardly counts in the mean squared error; geometric spacing puts half of the sample points below 5 m. And the default upper limit of the frequencies, $15/250\,\mathrm{m} = 0.06$ 1/m, is too low for the fine grain of the 5 m scale; the tuned fit allows up to 1 1/m. A mean over all distances hides where the error sits. A map of the error shows it: `sos.acf_2d` gives the target on a grid of displacements in the x-y plane, `sos.acf_approx` the ACF of a table on the same grid, and the cell plots their difference:

In [ ]:
d60 = np.arange(0.0, 60.1, 0.5)                                    # equally spaced, as both functions need
target_2d, d2 = qd.sos.acf_2d(two_scales(d60), d60)

fig, axs = plt.subplots(1, 2, figsize=(10, 3.8))
for ax, f, name in zip(axs, (freq_def, freq), ("default", "tuned")):
    approx_2d, _ = qd.sos.acf_approx(f, d60)                        # x-y plane is [:, :, 0]
    im = ax.imshow(approx_2d[:, :, 0] - target_2d, origin="lower", extent=[d2[0], d2[-1], d2[0], d2[-1]], cmap="RdBu_r", vmin=-0.15, vmax=0.15)
    ax.set_xlim(-25, 25); ax.set_ylim(-25, 25); ax.set_title(name); ax.set_xlabel("dx in m")
axs[0].set_ylabel("dy in m")
fig.colorbar(im, ax=axs, label="generator ACF minus target", shrink=0.9)
plt.show()

The fourth number, `acf_tail`, looks beyond the fitted range. Each sinusoid with a frequency near zero adds about $1/N$ to the ACF at every distance, so $n$ of them keep the ACF at about $n/N$ far away: values far apart are then still correlated. `freq_min` prevents this. Here both tails are small, because the default fit has only a few frequencies near zero; the exponential fits in "How many sinusoids" show a clear case. The tuned fit uses the lower limit of the presets, 0.001 1/m, which removes frequencies with wavelengths above 1 km.

The last check uses the generated values themselves. `sos.acf_estimate` evaluates one realization at random positions in a cube and correlates the values of all position pairs by distance, as in part 1. The cube must span many decorrelation distances of the longer scale, so the bins reach to 500 m; the estimate of one realization scatters, so the cell averages 20 realizations:

In [ ]:
bins = np.arange(2.0, 500.5, 2.0)                                  # distance bins; the cube has an edge of 0.8 × 500 m = 400 m
ph20 = qd.sos.init_phases(256, 20, 2)                               # 20 realizations
est = [qd.sos.acf_estimate(freq, ph20[:, r], bins, 1500, seed=r + 1) for r in range(20)]
acf_est = np.mean([e[0] for e in est], axis=0); dist_est = est[0][1]

plt.figure(figsize=(6.4, 3.6))
plt.plot(d_plot, two_scales(d_plot), "k--", lw=2, label="target")
plt.plot(d_plot, acfs["tuned"].mean(axis=0), "C0", label="generator (formula)")
plt.plot(dist_est, acf_est, "o", ms=3, color="C1", label="estimated from 20 fields")
plt.xlim(0, 150); plt.xlabel("distance d in m"); plt.ylabel("ACF"); plt.legend(); plt.grid(alpha=0.3)
plt.show()

The estimate follows the generator at short distances and lies somewhat below it at larger ones. The cause is the one described in part 1: the correlation in each bin is taken around the mean of the cube, which differs from the mean of the ensemble when the cube spans only a few lengths of the longer scale.

## Using your own frequencies

`sos.randn`, `sos.rand`, `sos.randi` and the channel-model functions take the name of a preset. Your own frequencies go to the low-level functions: `sos.init_phases` draws the phases from a seed, `sos.val` evaluates the generators at any positions, and `sos.map` on a grid, as in the first cell. Storing the frequencies and the seed is enough to make the same field again.

Two properties make one table go a long way. Dividing all frequencies by a factor stretches the field and its ACF by that factor [1, Sec. III], so one fit serves every decorrelation distance; this is how the presets are scaled. And multiplying one column of the table by a factor compresses the field along that axis: a table fitted as isotropic becomes one with a shorter correlation along z, as for a quantity that changes faster with height than along the ground. The cell checks both with the ACF formula:

In [ ]:
acf_of = lambda f, d_vec: np.cos(2 * np.pi * f @ np.asarray(d_vec)).mean()     # ACF of a table at one displacement
print(f"original table, 5 m along x:      {acf_of(freq, [5.0, 0, 0]):.3f}")
print(f"frequencies / 2, 10 m along x:    {acf_of(freq / 2, [10.0, 0, 0]):.3f}")
f_aniso = freq * np.array([1.0, 1.0, 5.0])                          # 5 times faster change along z
print(f"original table, 5 m along z:      {acf_of(freq, [0, 0, 5.0]):.3f}")
print(f"z column × 5, 1 m along z:        {acf_of(f_aniso, [0, 0, 1.0]):.3f}")

## Targets that are not ACFs

Not every curve that starts at 1 and falls can be an ACF. Bochner's theorem states the condition: a function is a valid ACF of a stationary field if and only if its spectrum is nowhere negative [2, Sec. 4.2.1, Thm. 4.1]. An equivalent test is that no correlation matrix built from it, for any set of positions, has a negative eigenvalue [2, Sec. 4.1]. The ACF of a sum of sinusoids is a sum of cosines with positive weights, a spectrum of positive lines, so it is always valid. A fit to an invalid target therefore cannot reach it; it ends at a valid ACF close to it.

An extreme example is the step "fully correlated up to 10 m, uncorrelated beyond". It is not valid in any dimension. Correlation 1 means equal values, and equality carries over: at positions 0, 6 and 12 m on a line, the values at 0 and 6 m must be equal and those at 6 and 12 m as well, so the values at 0 and 12 m cannot be uncorrelated. The correlation matrix of the three positions shows it:

In [ ]:
step = lambda d: (np.asarray(d) <= 10.0).astype(float)             # 1 up to 10 m, 0 beyond
p = np.array([0.0, 6.0, 12.0])
C = step(np.abs(p[:, None] - p[None, :]))
print(C)
print(f"determinant {np.linalg.det(C):.0f}, smallest eigenvalue {np.linalg.eigvalsh(C).min():.3f}")

What does the fit make of it? The cell fits the step in 3-D with 16, 64 and 256 sinusoids and shows the mean ACF over the 200 directions, and a map of the field with 256 sinusoids:

In [ ]:
d_step = np.arange(200) * 0.25                                      # 0 to 49.75 m
fig, axs = plt.subplots(1, 2, figsize=(10, 3.8))
axs[0].plot(d_step, step(d_step), "k--", lw=2, label="target")
for N, ls in ((16, ":"), (64, "-."), (256, "-")):
    f_step, _, _ = qd.sos.generate(step(d_step), d_step, N, n_dim=3, n_dir=20, seed=1)
    a = acf_along(f_step, d_step)
    axs[0].plot(d_step, a.mean(axis=0), ls, label=f"N = {N}")
    print(f"N = {N:3d}: rms error {np.sqrt(np.mean((a - step(d_step))**2)):.3f}, ACF at 10 m {np.interp(10, d_step, a.mean(axis=0)):.2f}, "
          f"lowest value {a.mean(axis=0).min():+.2f} at {d_step[a.mean(axis=0).argmin()]:.0f} m")
axs[0].set_xlabel("distance d in m"); axs[0].set_ylabel("ACF"); axs[0].legend(); axs[0].grid(alpha=0.3)
m = qd.sos.map(f_step, qd.sos.init_phases(256, 1, 4), x, x)[:, :, 0, 0]
im = axs[1].imshow(m, origin="lower", extent=[0, 300, 0, 300], cmap="viridis")
axs[1].set_xlabel("x in m"); axs[1].set_ylabel("y in m"); fig.colorbar(im, ax=axs[1], label="value", shrink=0.9)
plt.show()

All three fits give nearly the same curve and the same error: a smooth decay that passes 10 m at about one half and dips slightly below zero beyond. More sinusoids do not help, because the error is not an approximation error but the distance between the target and the nearest valid ACF the fit finds. The field looks like a smooth field with a decorrelation distance of about 10 m; nothing in it is "fully correlated" over 10 m.

Whether a target is valid can depend on the dimension. An example is the damped cosine $\rho(d) = e^{-d/20\,\mathrm{m}} \cos(2\pi d / 10\,\mathrm{m})$. Chuang et al. use this form in time, for the measured ACF of the fluctuations of the multipath parameters (path gain, delay, arrival angles) of each body part in a channel model for gesture recognition [3, Sec. III-B, eq. 2]. On a line it is valid (derived: its spectrum is the sum of two positive peaks). In two and three dimensions it is not: an isotropic ACF cannot fall below $-0.403$ in 2-D and $-0.217$ in 3-D [4, Sec. 2, eqs. 5-6], and this one reaches $-0.78$ near 4.9 m. In words: a position cannot be strongly anti-correlated with all positions on a circle around it, because those positions are close to each other and correlated among themselves. The cell fits it with 32 sinusoids in 1-D, with 64 and 256 in 2-D, and with 256 in 3-D:

In [ ]:
damped = lambda d: np.exp(-np.asarray(d) / 20.0) * np.cos(2 * np.pi * np.asarray(d) / 10.0)
d_fine = np.linspace(0.0, 10.0, 10001)
print(f"target: lowest value {damped(d_fine).min():.3f} at {d_fine[damped(d_fine).argmin()]:.2f} m")
plt.figure(figsize=(6.4, 3.6))
plt.plot(d_step, damped(d_step), "k--", lw=2, label="target")
fits_damped = {}
for n_dim, N in ((1, 32), (2, 64), (2, 256), (3, 256)):
    fits_damped[n_dim, N], _, _ = qd.sos.generate(damped(d_step), d_step, N, n_dim=n_dim, n_dir=20, seed=1)
    a = acf_along(fits_damped[n_dim, N], d_step)
    if N != 64:
        plt.plot(d_step, a.mean(axis=0), label=f"{n_dim}-D, N = {N}")
    print(f"{n_dim}-D, N = {N:3d}: rms error {np.sqrt(np.mean((a - damped(d_step))**2)):.3f}, lowest value {a.mean(axis=0).min():+.3f}")
plt.axhline(-0.403, color="gray", lw=0.8, ls=":", label="2-D bound -0.403")
plt.axhline(-0.217, color="gray", lw=0.8, ls="-.", label="3-D bound -0.217")
plt.xlim(0, 30); plt.xlabel("distance d in m"); plt.ylabel("ACF"); plt.legend(fontsize=8); plt.grid(alpha=0.3)
plt.show()
f_ring = np.linalg.norm(fits_damped[2, 256], axis=1)
print(f"2-D fit, N = 256: 80 % of the frequencies have |f| between {np.percentile(f_ring, 10):.3f} and {np.percentile(f_ring, 90):.3f} 1/m")

On a line the fit follows the target closely. In 2-D and 3-D the fits stop exactly at the bounds and cannot follow the oscillation, with 64 sinusoids as with 256. The 2-D fit shows how: it puts nearly all frequencies on a ring of one radius, and the ACF of such a ring is the Bessel function $J_0$, whose minimum is the bound (derived; [4, Sec. 2] derives the bound from the same function). For an isotropic quantity in a plane or in space, a target like this cannot be realized by any method, not only by a sum of sinusoids.

The triangle $\rho(d) = 1 - d/a$ up to $a$ is the simplest of the compactly supported ACFs of [2, Sec. 4.2.1, eq. 4.21] (dimension 1), whose validity is restricted to a maximum dimension. In 2-D its spectrum is only slightly negative (derived: its minimum is about −2 % of the value at zero frequency), and a 2-D fit with 256 sinusoids follows it within the error printed here:

In [ ]:
tri = np.maximum(1.0 - d_step / 10.0, 0.0)
f_tri, _, _ = qd.sos.generate(tri, d_step, 256, n_dim=2, n_dir=20, seed=1)
print(f"triangle, 2-D, N = 256: rms error {np.sqrt(np.mean((acf_along(f_tri, d_step) - tri)**2)):.3f}")

## How many sinusoids

How fast does the error fall with the number of sinusoids, and what happens beyond the fitted range? The cell fits the exponential ACF ($d_0$ = 10 m, sample points up to 49.75 m, i.e. 5 $d_0$, and the lower limit $0.05/d_0$ of the presets) in 3-D with 8 to 256 sinusoids, and adds the presets `Exp128` to `Exp1024`. Two more fits with 256 sinusoids show the effect of the settings: one with 300 test directions, as the presets, and one without a lower limit. For each table the cell computes the rms error along the 200 directions, inside the fitted range and between 5 and 30 $d_0$, where the target is nearly 0, and the mean ACF in that range:

In [ ]:
d_in, d_out = np.linspace(0.0, 49.75, 200), np.linspace(49.75, 300.0, 500)
exp_acf = lambda d: np.exp(-d / 10.0)
fit = lambda N, **kw: qd.sos.generate(exp_acf(d_step), d_step, N, n_dim=3, seed=1, **kw)[0]
tables = [("fit", f"fit, N = {N}", fit(N, n_dir=20, freq_min=0.005)) for N in (8, 16, 32, 64, 128, 256)]
tables += [("preset", name, qd.sos.preset(name, 10.0)[0]) for name in ("Exp128", "Exp256", "Exp512", "Exp1024")]
tables += [("other", "fit, N = 256, 300 test directions", fit(256, n_dir=300, freq_min=0.005)),
           ("other", "fit, N = 256, no lower limit", fit(256, n_dir=20))]
err = []
for kind, label, f in tables:
    a_out = acf_along(f, d_out)
    e_in, e_out = np.sqrt(np.mean((acf_along(f, d_in) - exp_acf(d_in))**2)), np.sqrt(np.mean((a_out - exp_acf(d_out))**2))
    err.append((kind, f.shape[0], e_in, e_out))
    print(f"{label:34s} inside {e_in:.3f}   beyond {e_out:.3f}   mean beyond {a_out.mean():+.3f}   1/sqrt(2N) {1/np.sqrt(2*f.shape[0]):.3f}   "
          f"calc_mse {qd.sos.calc_mse(f, exp_acf(d_step), d_step, n_dir=100)[0]:4.1f} dB")

In [ ]:
Ns = np.array([8, 1024])
plt.figure(figsize=(6.4, 3.8))
for kind, mk in (("fit", "o"), ("preset", "s")):
    e = np.array([r[1:] for r in err if r[0] == kind])               # columns N, inside, beyond
    plt.loglog(e[:, 0], e[:, 1], mk + "-", color="C0", label=f"inside the fitted range ({kind})")
    plt.loglog(e[:, 0], e[:, 2], mk + "-", color="C1", label=f"beyond it ({kind})")
plt.loglog(Ns, 1 / np.sqrt(2 * Ns), "k--", lw=1, label="1/sqrt(2N)")
plt.xlabel("number of sinusoids N"); plt.ylabel("rms error of the ACF"); plt.legend(fontsize=8); plt.grid(alpha=0.3, which="both")
plt.show()

Both errors fall as $1/\sqrt{N}$: each doubling of $N$ reduces the rms error by a factor of $\sqrt{2}$, which is 3 dB of the mean squared error that `sos.calc_mse` reports, as found in [1, Sec. V, Fig. 3] for the error inside the range. The presets are a little more accurate than the fits with 20 test directions at the same $N$; with 300 test directions, as for the presets, a fit reaches them. The fit without a lower limit has a plateau beyond the range: its mean ACF there stays above zero, as described in "Judging a fit".

Beyond the fitted range the error is the ACF of the generator itself, since the target is nearly 0. It does not decay: its rms is $1/\sqrt{2N}$, 0.044 for 256 sinusoids (derived: the mean of $\cos^2$ is $1/2$ for each of $N$ distinct frequencies). This residual belongs to the table: all realizations drawn with these frequencies share it, so an average over many of them converges to it, not to 0. It differs from direction to direction, though, so its average over directions is much smaller. A random rotation of the table for each realization would turn the ensemble ACF into this average. For a uniformly random rotation the average has a closed form, $\frac{1}{N}\sum_n \sin(2\pi |\mathbf{f}_n| d)/(2\pi |\mathbf{f}_n| d)$ (derived); the cell compares it with the residual along single directions:

In [ ]:
f256, _, _ = qd.sos.preset("Exp256", 10.0)
a_single = acf_along(f256, d_out)                                   # [200 directions, n_d]
a_rot = np.sinc(2 * np.outer(d_out, np.linalg.norm(f256, axis=1))).mean(axis=1)   # average over all rotations; np.sinc(x) = sin(pi x)/(pi x)
print(f"Exp256 beyond 5 d0: rms along single directions {np.sqrt(np.mean(a_single**2)):.3f}, rms of the average over rotations {np.sqrt(np.mean(a_rot**2)):.4f}")
for lo, hi in ((50, 100), (100, 200), (200, 300)):
    sel = (d_out >= lo) & (d_out < hi)
    print(f"   average over rotations, {lo} to {hi} m: rms {np.sqrt(np.mean(a_rot[sel]**2)):.4f}")

The average falls to about a tenth of the residual and decays with distance. In 1-D, there are no directions to average over, and the fit does not control the ACF beyond the range at all; the residual can be larger than $1/\sqrt{2N}$. The fit may place several sinusoids at nearly the same frequency, closer together than it can tell apart over the fitted range; this costs nothing inside the range, but beyond it they add up like one sinusoid. The 1-D fit of the damped cosine above shows it:

In [ ]:
f1 = fits_damped[1, 32][:, 0]
d_far = np.linspace(50.0, 1000.0, 20000)
a1 = np.cos(2 * np.pi * np.outer(d_far, f1)).mean(axis=1)
print(f"1-D, N = 32, 50 to 1000 m: rms {np.sqrt(np.mean(a1**2)):.3f} (1/sqrt(2N) = {1/np.sqrt(64):.3f}); acf_tail {np.round(qd.sos.calc_mse(fits_damped[1, 32], damped(d_step), d_step)[3], 2)}")

For a 1-D generator, check `acf_tail` and let the sample points cover all distances at which the values will be compared.

## The slope at the origin

The exponential ACF has a kink at $d = 0$: it falls with the slope $-1/d_0$ from the start, which makes the field rough at all scales. A sum of $N$ sinusoids has a smooth ACF, flat at the origin, so no finite table reproduces the kink [1, Sec. V]. The fit trades it against the error elsewhere; more sinusoids bring the ACF closer to the kink. One consequence is the number of zero crossings per meter along a track. For a field with Normal values it is $\frac{1}{\pi}\sqrt{-\rho''(0)}$, or in terms of the spectrum twice the root of the mean squared frequency [5, Sec. 3.3, eq. 3.3-11]; for a sum of sinusoids along x this is $2\sqrt{\overline{f_x^2}}$, with the mean over the sinusoids, as far as the values are close to Normal. For the exponential ACF itself it is infinite [5, Sec. 3.3]. The cell shows the first meters of the ACF of the `Exp` presets and counts the zero crossings along a 10 km track, for 5 seeds:

In [ ]:
d_near = np.linspace(0.0, 3.0, 301)
track = np.zeros((100001, 3)); track[:, 0] = np.linspace(0.0, 10000.0, 100001)    # 10 km along x, 10 cm steps
plt.figure(figsize=(6.4, 3.6))
plt.plot(d_near, np.exp(-d_near / 10.0), "k--", lw=2, label="target exp(-d/10 m)")
for name in ("Exp128", "Exp256", "Exp512", "Exp1024", "Comb256"):
    f_p, _, _ = qd.sos.preset(name, 10.0)
    if name.startswith("Exp"):
        plt.plot(d_near, acf_along(f_p, d_near).mean(axis=0), label=name)
    v = qd.sos.val(f_p, qd.sos.init_phases(f_p.shape[0], 5, 3), track)          # [positions, 5 seeds]
    counts = np.sum(np.sign(v[1:]) != np.sign(v[:-1]), axis=0) / 100.0          # crossings per 100 m for each seed
    print(f"{name:8s} zero crossings per 100 m: formula {200 * np.sqrt(np.mean(f_p[:, 0]**2)):5.1f}, counted {counts.mean():5.1f} ± {counts.std():.1f}")
plt.xlabel("distance d in m"); plt.ylabel("ACF"); plt.legend(); plt.grid(alpha=0.3)
plt.show()

The larger the preset, the closer its ACF comes to the kink, and the more often the field crosses zero; the formula and the count agree within the scatter between seeds, printed after the ± sign. A quantity that depends on the roughness of the field at short distances, such as level-crossing rates or the length of fades along a route, therefore depends on the number of sinusoids. `Comb` avoids this: it is smooth at the origin by design, and its crossing rate is the same for all sizes [1, Sec. V].

## Few sinusoids

With few sinusoids, the values are not Normal distributed; part 1 shows the histogram. What matters in practice are the tails: a sum of $N$ cosines with amplitude $\sqrt{2/N}$ never exceeds $\sqrt{2N}$, and rare low values, such as deep fades below a threshold, occur less often than for Normal values. The cell counts them over 400 000 realizations at one position; at one position only the phases matter, so any table gives the same values:

In [ ]:
from math import erfc, sqrt
f_p, _, _ = qd.sos.preset("Comb256", 10.0)
print(f"Normal:  P(s < -2.33) = {0.5 * erfc(2.33 / sqrt(2)):.4f}   P(s < -3) = {0.5 * erfc(3 / sqrt(2)):.5f}")
for N in (4, 8, 16, 32, 64, 128):
    v = np.concatenate([qd.sos.val(f_p[:N], qd.sos.init_phases(N, 50000, 10 + c), np.zeros((1, 3))).ravel() for c in range(8)])
    print(f"N = {N:3d}:  P(s < -2.33) = {np.mean(v < -2.33):.4f}   P(s < -3) = {np.mean(v < -3):.5f}   max |s| = {np.abs(v).max():.2f} (bound {sqrt(2 * N):.2f})")

With 4 sinusoids no value falls below −3, against 0.135 % for Normal values; with 8 about half of them do, with 32 about 90 %. At the 1 % level (−2.33) the share is within 15 % of the Normal value from 8 sinusoids on. Simulations that count rare events need enough sinusoids, or a check of the tails like this one.

## A traveling sea surface

The frequencies do not have to come from a fit. The ACF of a sum of sinusoids is the Fourier transform of its line spectrum, so frequencies drawn at random from a target spectrum, used as a probability density, give a generator whose ACF is the target on average over draws; the same construction underlies random Fourier features in machine learning [6, Sec. 3]. One draw has an error of about $1/\sqrt{2N}$ at all distances, more than a good fit inside its range. In return it works where `sos.generate` cannot help: the fit makes isotropic tables, while many fields are not isotropic.

A wind-driven sea surface is an example. Tessendorf builds its height from waves on a lattice of wave vectors, with Gaussian random amplitudes whose variance follows the Phillips spectrum [7, Secs. 4.3-4.4, eqs. 36, 40, 42]; here, the wave vectors are drawn from that spectrum instead, with equal amplitudes. The spectrum is $P(\mathbf{k}) \propto e^{-1/(kL)^2}\, |\hat{\mathbf{k}} \cdot \hat{\mathbf{w}}|^2 / k^4$, with $L = V^2/g$ for the wind speed $V$ and the wind direction $\hat{\mathbf{w}}$ [7, eq. 40], times a factor $e^{-k^2 \ell^2}$ that suppresses waves shorter than $\ell$ [7, eq. 41]. Each wave travels with the angular frequency of deep water, $\omega = \sqrt{g k}$ [7, eq. 31]. With time as the third coordinate, a wave $\cos(\mathbf{k} \cdot \mathbf{x} - \omega t + \varphi)$ is a sinusoid with the frequency vector $(k_x, k_y, -\omega)/2\pi$, so `sos.map` evaluates the surface at any place and time. The radial part of the spectrum can be drawn directly: with $u = 1/(kL)^2$, the density of $k$ in the plane becomes an exponential distribution of $u$ (derived). The small-wave factor and the direction are applied by rejection: a draw is kept with the probability $e^{-k^2\ell^2} \cos^2\alpha$, with the angle $\alpha$ to the wind. Keeping only waves that move with the wind is a choice made here.

In [ ]:
g, V, ell, N = 9.81, 10.0, 0.5, 256                  # gravity in m/s², wind speed in m/s (along x), small-wave cutoff in m, waves
L = V**2 / g
rng = np.random.default_rng(4)
k, alpha = np.empty(0), np.empty(0)
while k.size < N:
    kk = 1.0 / (L * np.sqrt(rng.exponential(size=4 * N)))          # wave numbers in rad/m from the radial part of the spectrum
    angle = rng.uniform(-np.pi / 2, np.pi / 2, kk.size)             # angle to the wind direction in rad, downwind half only
    keep = rng.uniform(size=kk.size) < np.exp(-(kk * ell)**2) * np.cos(angle)**2
    k, alpha = np.r_[k, kk[keep]], np.r_[alpha, angle[keep]]
k, alpha = k[:N], alpha[:N]
omega = np.sqrt(g * k)                                              # deep-water dispersion, rad/s
freq_sea = np.c_[k * np.cos(alpha), k * np.sin(alpha), -omega] / (2 * np.pi)   # [N, 3]: cycles per m in x and y, cycles per s in t
print(f"median wavelength {np.median(2 * np.pi / k):.0f} m, median period {np.median(2 * np.pi / omega):.1f} s, median phase speed {np.median(omega / k):.1f} m/s")

Two frames 2 s apart, scaled to an rms height of 0.5 m:

In [ ]:
xs = np.arange(0.0, 400.0, 1.0)                                    # grid points in m
h = 0.5 * qd.sos.map(freq_sea, qd.sos.init_phases(N, 1, 8), xs, xs, np.array([0.0, 2.0]))[:, :, :, 0]   # [y, x, t] in m

fig, axs = plt.subplots(1, 2, figsize=(10, 3.8))
for i, ax in enumerate(axs):
    im = ax.imshow(h[:, :, i], origin="lower", extent=[0, 400, 0, 400], cmap="Blues_r", vmin=-1.5, vmax=1.5)
    ax.set_title(f"t = {2 * i} s"); ax.set_xlabel("x in m (wind direction)")
axs[0].set_ylabel("y in m")
fig.colorbar(im, ax=axs, label="surface height in m", shrink=0.9)
plt.show()
shift = [np.mean(h[:, :xs.size - s, 0] * h[:, s:, 1]) for s in range(40)]   # frame 1 against frame 2 shifted by s grid points along x
print(f"rms height of the first frame {h[:, :, 0].std():.2f} m; best match after a shift of {np.argmax(shift):.0f} m along x in 2 s")

The crests are short, because the waves come from a broad range of directions, and run mostly across the wind. The pattern moves with the wind by the printed shift in 2 s, close to the median phase speed of the waves. The rms height of this frame is close to the 0.5 m of the ensemble; a smaller frame, with only a few wavelengths of the longest waves, would scatter more around it. Unlike a surface made by FFT on a grid, which repeats itself after the size of the grid [7, Sec. 4.3], this one has no period in space or time, and it can be evaluated at the positions of a floating body or a sensor alone. It shares the limits of the height-field model of [7, Secs. 4.3-4.4]: linear waves with Normal heights and rounded crests. The sharper "choppy" crests of [7, Sec. 4.6], made by a horizontal displacement of the surface points, are not applied here, and breaking waves are outside both.

## Limits of the method

The sections above show where a sum of $N$ equal-amplitude sinusoids reaches its limits:

- The ACF is approximated, with an rms error that falls as $1/\sqrt{N}$, and only over the fitted range; beyond it, a residual of rms $1/\sqrt{2N}$ remains, shared by all realizations of a table ("How many sinusoids").
- A target that is not a valid ACF is replaced by a valid one; validity can depend on the dimension, and for an isotropic field no method can do better ("Targets that are not ACFs").
- The kink of the exponential ACF at the origin is not reproduced, and quantities that depend on the roughness at short distances depend on $N$ ("The slope at the origin").
- With few sinusoids, rare extreme values are missing ("Few sinusoids").
- The cost of each evaluation grows with $N$ (part 1).
- `sos.generate` fits isotropic tables; other fields need their own frequencies ("A traveling sea surface"), or a scaled table ("Using your own frequencies"). In 1-D it leaves the ACF beyond the fitted range uncontrolled.
- The field is stationary and unconditioned: it cannot be made to pass through measured values at given positions.

## Summary

- The frequencies are the only free parameters of a sum-of-sinusoids generator; their density takes the place of the spectrum.
- `sos.generate` fits them to an ACF of your own; the sample points decide where the fit is accurate, `freq_max` how fine it can be, `freq_min` what happens far away, and the number of test directions how evenly it fits all directions.
- Judge a table by its ACF, not only by `sos.calc_mse`: plot it against the target, map the error with `sos.acf_approx` and `sos.acf_2d`, check `acf_tail`, and estimate it from generated values with `sos.acf_estimate`.
- Your own tables are used with `sos.init_phases`, `sos.val` and `sos.map`; dividing them by a factor changes the decorrelation distance, scaling a column makes the field anisotropic.
- Frequencies drawn from a spectrum give generators that no fit provides, including fields in space and time.

## References

[1] S. Jaeckel, L. Raschkowski, F. Burkhardt, L. Thiele, "Efficient sum-of-sinusoids-based spatial consistency for the 3GPP new-radio channel model", IEEE Globecom Workshops, 2018. [PDF](publications/2018_sum_of_sinusoids.pdf), [doi:10.1109/GLOCOMW.2018.8644265](https://doi.org/10.1109/GLOCOMW.2018.8644265).

[2] C. E. Rasmussen, C. K. I. Williams, *Gaussian Processes for Machine Learning*, MIT Press, 2006, chapter 4. [Free online](https://gaussianprocess.org/gpml/chapters/), [doi:10.7551/mitpress/3206.001.0001](https://doi.org/10.7551/mitpress/3206.001.0001).

[3] J. Chuang, R. Caromi, J. Senic, S. Berweger, N. Varshney, J. Wang, C. Lai, A. Bodi, W. Sloane, C. Gentile, N. Golmie, "Quasi-deterministic channel propagation model for human sensing: gesture recognition use case", IEEE Open Journal of Antennas and Propagation 5(3), 557-572, 2024. [doi:10.1109/OJAP.2024.3371834](https://doi.org/10.1109/OJAP.2024.3371834).

[4] P. Schneider, J. Hartlap, "Constrained correlation functions", 2009. [arXiv:0905.0577](https://arxiv.org/abs/0905.0577). Published in Astronomy & Astrophysics 504(3), 705-717, 2009, [doi:10.1051/0004-6361/200912424](https://doi.org/10.1051/0004-6361/200912424).

[5] S. O. Rice, "Mathematical analysis of random noise" (Parts III and IV), Bell Syst. Tech. J. 24(1), 46-156, 1945. [doi:10.1002/j.1538-7305.1945.tb00453.x](https://doi.org/10.1002/j.1538-7305.1945.tb00453.x).

[6] A. Rahimi, B. Recht, "Random features for large-scale kernel machines", Advances in Neural Information Processing Systems 20, 2007. [Online](https://papers.nips.cc/paper/2007/hash/013a006f03dbc5392effeb8f18fda755-Abstract.html).

[7] J. Tessendorf, "Simulating ocean water", course notes, 1999-2004. [PDF](https://jtessen.people.clemson.edu/reports/papers_files/coursenotes2004.pdf).